# PNA Blinatumomab Co-culture: Quality Control & Initial Analysis

Pipeline: pixelator v0.21.3 PNA (`--strategy paired`)
Panel: proxiome-immuno-156-FMC63 (159 markers)

In [ ]:
import sys
sys.path.insert(0, '/home/projects/nyosef/zvise/PixelGen/PixelGen')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pixelator import read_pna

# --- Global figure style ---
sns.set_style("whitegrid")
sc.settings.set_figure_params(dpi=120, frameon=False, fontsize=12)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "sans-serif",
})

RESULTS_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/results")
MODEL_DIR = RESULTS_DIR / "models" / "scvi_baseline"

# Color palettes
COND_PALETTE = {"Mock": "#4c72b0", "Blinatumomab": "#dd8452"}
TIME_PALETTE = {"6h": "#55a868", "48h": "#c44e52"}
SYSTEM_PALETTE = {
    "healthy B + healthy T": "#4878d0",
    "NALM-6 + healthy T": "#ee854a",
    "patient B + patient T": "#6acc65",
    "NALM-6 + patient T": "#d65f5f",
}

# Sample metadata
sample_meta = {
    "S001": {"time": "6h",  "condition": "Mock",           "target": "healthy B",  "tcells": "healthy T"},
    "S002": {"time": "6h",  "condition": "Blinatumomab",   "target": "healthy B",  "tcells": "healthy T"},
    "S003": {"time": "48h", "condition": "Mock",           "target": "healthy B",  "tcells": "healthy T"},
    "S004": {"time": "48h", "condition": "Blinatumomab",   "target": "healthy B",  "tcells": "healthy T"},
    "S005": {"time": "6h",  "condition": "Mock",           "target": "NALM-6",     "tcells": "healthy T"},
    "S006": {"time": "6h",  "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "healthy T"},
    "S007": {"time": "48h", "condition": "Mock",           "target": "NALM-6",     "tcells": "healthy T"},
    "S008": {"time": "48h", "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "healthy T"},
    "S009": {"time": "6h",  "condition": "Mock",           "target": "patient B",  "tcells": "patient T"},
    "S010": {"time": "6h",  "condition": "Blinatumomab",   "target": "patient B",  "tcells": "patient T"},
    "S011": {"time": "48h", "condition": "Mock",           "target": "patient B",  "tcells": "patient T"},
    "S012": {"time": "48h", "condition": "Blinatumomab",   "target": "patient B",  "tcells": "patient T"},
    "S013": {"time": "6h",  "condition": "Mock",           "target": "NALM-6",     "tcells": "patient T"},
    "S014": {"time": "6h",  "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "patient T"},
    "S016": {"time": "48h", "condition": "Blinatumomab",   "target": "NALM-6",     "tcells": "patient T"},
}

## 1. Load Data

In [ ]:
# Load all available samples
pxl_files = []
for sample_id in sample_meta:
    pxl_path = RESULTS_DIR / sample_id / "layout" / "layout" / f"{sample_id}.layout.pxl"
    if pxl_path.exists():
        pxl_files.append(pxl_path)
        print(f"  {sample_id}: {pxl_path.stat().st_size / 1e9:.1f} GB")
    else:
        print(f"  {sample_id}: NOT READY")

print(f"\nLoading {len(pxl_files)} samples...")
pg = read_pna(pxl_files)
print("Done.")

In [ ]:
# Add experimental metadata to obs
adata = pg.adata()
for sample_id, meta in sample_meta.items():
    mask = adata.obs["sample"] == sample_id
    for key, val in meta.items():
        adata.obs.loc[mask, key] = val

# Combined label for plotting
adata.obs["group"] = adata.obs["condition"] + "_" + adata.obs["time"]
adata.obs["cell_system"] = adata.obs["target"] + " + " + adata.obs["tcells"]

print(adata)
print(f"\nSamples: {adata.obs['sample'].nunique()}")
print(f"Cells per sample:")
print(adata.obs["sample"].value_counts().sort_index().to_string())

## 2. Quality Control Metrics

Key QC metrics from pixelator:
- **n_umi**: total unique molecular identifiers (proxy for cell size / capture quality)
- **n_edges**: number of edges in the cell graph
- **n_antibodies**: number of distinct antibodies detected
- **tau / tau_type**: cell-type classification metric (normal vs outlier)
- **reads_in_component**: total sequencing reads assigned to this cell
- **isotype_fraction**: fraction of reads from isotype controls (background noise)

In [ ]:
# Derive additional QC metrics
adata.obs["reads_per_umi"] = adata.obs["reads_in_component"] / adata.obs["n_umi"]
adata.obs["n_umi_log10"] = np.log10(adata.obs["n_umi"])
adata.obs["n_edges_log10"] = np.log10(adata.obs["n_edges"])

# Summary table
qc_cols = ["n_umi", "n_edges", "n_antibodies", "reads_per_umi", "isotype_fraction", "tau"]
print(adata.obs[qc_cols].describe().round(2).to_string())

In [ ]:
# Molecule rank plot per sample (cell calling)
fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.cm.tab20(np.linspace(0, 1, 15))
for i, sample_id in enumerate(sorted(adata.obs["sample"].unique())):
    mask = adata.obs["sample"] == sample_id
    umis = adata.obs.loc[mask, "n_umi"].sort_values(ascending=False).values
    ax.plot(range(1, len(umis) + 1), umis, label=sample_id, alpha=0.8, lw=1.5, color=cmap[i])
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Cell rank")
ax.set_ylabel("UMI count")
ax.set_title("Molecule Rank Plot ", fontweight="bold")
ax.axhline(y=25000, color="grey", ls="--", lw=1, alpha=0.6, label="MIN_UMI = 10k")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8, frameon=True)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# QC metric distributions per sample (violin + strip)
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
metrics = ["n_umi_log10", "n_edges_log10", "n_antibodies", "reads_per_umi", "isotype_fraction", "tau"]
titles = ["log\u2081\u2080(UMI)", "log\u2081\u2080(Edges)", "Antibodies detected", "Reads / UMI", "Isotype fraction", "Tau"]

for ax, metric, title in zip(axes.flat, metrics, titles):
    sns.violinplot(
        data=adata.obs, x="sample", y=metric, order=sorted(adata.obs["sample"].unique()),
        inner=None, color="lightsteelblue", alpha=0.6, ax=ax, linewidth=0.5,
    )
    sns.stripplot(
        data=adata.obs, x="sample", y=metric, order=sorted(adata.obs["sample"].unique()),
        color="steelblue", alpha=0.15, size=1.5, jitter=True, ax=ax,
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("QC Metrics by Sample", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# tau_type distribution per sample
tau_counts = adata.obs.groupby(["sample", "tau_type"]).size().unstack(fill_value=0)
tau_frac = tau_counts.div(tau_counts.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

tau_counts.plot(kind="bar", stacked=True, colormap="Set2", ax=axes[0], edgecolor="white", lw=0.5)
axes[0].set_title("Cell count by tau_type", fontweight="bold")
axes[0].set_ylabel("Number of cells")
axes[0].set_xlabel("")
axes[0].legend(title="tau_type", frameon=True)

tau_frac.plot(kind="bar", stacked=True, colormap="Set2", ax=axes[1], edgecolor="white", lw=0.5)
axes[1].set_title("Fraction by tau_type", fontweight="bold")
axes[1].set_ylabel("Fraction")
axes[1].set_xlabel("")
axes[1].legend(title="tau_type", frameon=True)

plt.tight_layout()
plt.show()

print("\ntau_type counts:")
print(tau_counts.to_string())

## 3. Cell Filtering

Filter cells based on:
- `tau_type == "normal"` (remove outlier cells)
- Minimum UMI threshold (inspect rank plot above to decide)

In [ ]:
# Filter cells
MIN_UMI = 25000  # adjust based on rank plot

keep = (
    (adata.obs["tau_type"] == "normal") &
    (adata.obs["n_umi"] >= MIN_UMI)
)
print(f"Before filtering: {adata.n_obs} cells")
print(f"  tau_type != normal: {(adata.obs['tau_type'] != 'normal').sum()}")
print(f"  n_umi < {MIN_UMI}: {(adata.obs['n_umi'] < MIN_UMI).sum()}")

adata_filtered = adata[keep].copy()
print(f"After filtering: {adata_filtered.n_obs} cells")
print(f"\nFiltered cells per sample:")
print(adata_filtered.obs["sample"].value_counts().sort_index().to_string())

## 4. Abundance Normalization

In [ ]:
# CLR normalization (centered log-ratio)
from pixelator.common.statistics import clr_transformation

raw_df = adata_filtered.to_df()
adata_filtered.layers["clr"] = clr_transformation(raw_df, axis=0, non_negative=True)

# Log1p for comparison
adata_filtered.layers["log1p"] = np.log1p(raw_df)

print("Available layers:", list(adata_filtered.layers.keys()))
print("CLR shape:", adata_filtered.layers["clr"].shape)

## 5. Dimensionality Reduction & Integration

Use **scVI** with sample as batch key for learned batch correction via variational inference.

In [ ]:
# --- scVI: batch-corrected latent space ---
scvi.model.SCVI.setup_anndata(adata_filtered, layer="clr", batch_key="sample")

if MODEL_DIR.exists():
    print(f"Loading cached scVI model from {MODEL_DIR}")
    scvi_model = scvi.model.SCVI.load(str(MODEL_DIR), adata=adata_filtered)
else:
    print("Training scVI...")
    scvi_model = scvi.model.SCVI(adata_filtered, n_latent=20, n_hidden=128, n_layers=2)
    scvi_model.train(max_epochs=400, early_stopping=True, early_stopping_patience=20)
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    scvi_model.save(str(MODEL_DIR), overwrite=True)
    print(f"Model saved to {MODEL_DIR}")

print(f"Trained for {scvi_model.history['elbo_train'].shape[0]} epochs")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ["elbo_train", "elbo_validation"], ["Train ELBO", "Validation ELBO"]):
    sns.lineplot(data=scvi_model.history[key], ax=ax, color="steelblue")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("ELBO")
plt.tight_layout()
plt.show()

In [ ]:
# Extract scVI latent, compute UMAP and cluster
adata_filtered.obsm["z_scvi"] = scvi_model.get_latent_representation()
print(f"scVI latent: {adata_filtered.obsm['z_scvi'].shape}")

# scVI-based UMAP
sc.pp.neighbors(adata_filtered, use_rep="z_scvi", n_neighbors=15)
sc.tl.umap(adata_filtered, min_dist=0.3)

# Leiden clustering
sc.tl.leiden(adata_filtered, resolution=0.5, key_added="leiden", random_state=42)
print(f"leiden: {adata_filtered.obs['leiden'].nunique()} clusters")

In [ ]:
# UMAP colored by experimental variables
color_keys = ["sample", "condition", "time", "cell_system", "leiden"]
palettes = [None, COND_PALETTE, TIME_PALETTE, SYSTEM_PALETTE, None]
titles = ["Sample", "Condition", "Timepoint", "Cell System"]

fig, axes = plt.subplots(1, 4, figsize=(40, 10))
for ax, ckey, pal, title in zip(axes, color_keys, palettes, titles):
    sc.pl.umap(
        adata_filtered, color=ckey, palette=pal,
        title=title, ax=ax, show=False,
        frameon=False, size=20, alpha=0.6,
    )
plt.suptitle("scVI UMAP (batch-corrected)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Marker Expression

Visualize key lineage markers on UMAP to identify cell types:
- **T cells**: CD3, CD4, CD8, CD45
- **B cells**: CD19, CD20, CD22, CD79B
- **Other**: CD56 (NK), CD14 (monocytes), FMC63 (CAR construct)

In [ ]:
# Use CLR layer for marker visualization, scVI UMAP
adata_filtered.X = adata_filtered.layers["clr"]

# Organized marker panels
marker_groups = {
    "T cell": ["CD3e", "TCRab", "CD5", "CD7", "CD2"],
    "CD4 T": ["CD4", "CD45RO", "CD45RA", "CD127"],
    "CD8 T": ["CD8", "CD57", "KLRG1", "CD279"],
    "B cell": ["CD19", "CD20", "CD22", "CD24", "IgM", "IgD", "CD27"],
    "NK": ["CD56", "CD94",],
    "Myeloid": ["CD14", "CD11c", "CD64", "CD33", "HLA-DR", "CD163","FMC63"],
}

# Filter to markers present in panel
marker_groups_filtered = {}
for group, mks in marker_groups.items():
    present = [m for m in mks if m in adata_filtered.var_names]
    missing = [m for m in mks if m not in adata_filtered.var_names]
    if present:
        marker_groups_filtered[group] = present
    if missing:
        print(f"{group} — missing: {missing}")

# Flat list for UMAP
markers = list(dict.fromkeys(m for mks in marker_groups_filtered.values() for m in mks))
print(f"\nPlotting {len(markers)} markers on scVI UMAP")

# sc.pl.umap(adata_filtered, color=markers, ncols=5, frameon=False, vmax="p99", size=6)

In [ ]:
# Dotplot: grouped markers by cluster
sc.pl.dotplot(
    adata_filtered,
    var_names=marker_groups_filtered,
    groupby="leiden",
    standard_scale="var",
    title="Marker expression by cluster (CLR, scaled)",
    figsize=(20, 6),
)

In [ ]:
# Assign cell types based on dotplot inspection
# Edit this dict after examining the dotplot above
cluster_to_celltype = {
    "0": "CD4",
    "1": "B",
    "2": "B",
    "3": "CD8",
    "4": "CD4", 
    "5": "CD4",
    "6": "CD4",
    "7": "B",
    "8": "B",
     "9": "B",
    "10": "CD8",
    "11": "B",
    "12": "B",
    "13": "CD8",
    "14": "CD8", 
    "15": "CD8",
    "16": "B",
   
     
}

adata_filtered.obs["cell_type"] = (
    adata_filtered.obs["leiden"]
    .map(cluster_to_celltype)
    .fillna("Unknown")
    .astype("category")
)

print(adata_filtered.obs["cell_type"].value_counts().to_string())

sc.pl.umap(adata_filtered, color=["cell_type",'cell_system','condition','time'], frameon=False, size=10,
           title="Cell type annotation")

## 7. Differential Abundance

Compare cell type composition:
1. **Within each cell system**: Mock vs Blinatumomab effect on cell type proportions
2. **B/T ratio**: across cell systems and conditions

In [ ]:
# --- 7a. Cell type composition over time per cell system ---
# x = timepoint (6h → 48h), y = fraction, lines connect conditions
ct_counts = adata_filtered.obs.groupby(["sample", "cell_type"], observed=True).size().unstack(fill_value=0)
ct_frac = ct_counts.div(ct_counts.sum(axis=1), axis=0)

sample_info = adata_filtered.obs.groupby("sample", observed=True)[["condition", "time", "cell_system"]].first()
ct_frac = ct_frac.join(sample_info)
ct_frac["time_h"] = ct_frac["time"].astype(str).str.replace("h", "").astype(int)

ct_cols = [c for c in ct_frac.columns if c not in ["condition", "time", "cell_system", "time_h"]]
ct_melt = ct_frac.reset_index().melt(
    id_vars=["sample", "condition", "time", "time_h", "cell_system"],
    value_vars=ct_cols, var_name="cell_type", value_name="fraction",
)

systems = sorted(ct_melt["cell_system"].unique())
cell_types = sorted(ct_melt["cell_type"].unique())
n_sys = len(systems)

fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, 5), sharey=True)
if n_sys == 1:
    axes = [axes]

for ax, sys in zip(axes, systems):
    sub = ct_melt[ct_melt["cell_system"] == sys]
    for ct in cell_types:
        for cond, ls in zip(["Mock", "Blinatumomab"], ["-", "--"]):
            d = sub[(sub["cell_type"] == ct) & (sub["condition"] == cond)]
            if d.empty:
                continue
            means = d.groupby("time_h")["fraction"].mean()
            ax.plot(means.index, means.values, marker="o", ls=ls, label=f"{ct} ({cond})", markersize=6)
            # Also scatter individual points
            ax.scatter(d["time_h"], d["fraction"], alpha=0.3, s=20, zorder=1)

    ax.set_title(sys, fontweight="bold", fontsize=12)
    ax.set_xlabel("Time (h)")
    ax.set_xticks([6, 48])
    ax.set_ylabel("Fraction of cells" if ax == axes[0] else "")
    sns.despine(ax=ax)

# Single legend outside
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, title="Cell type (condition)", frameon=True,
           bbox_to_anchor=(1.02, 0.5), loc="center left", fontsize=9)
fig.suptitle("Cell type composition over time — solid=Mock, dashed=Blinatumomab",
             fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# --- 7b. B/T ratio over time per cell system ---
B_LINEAGE = {"B"}
T_LINEAGE = {"CD4", "CD8"}

obs = adata_filtered.obs
bt_df = obs.groupby("sample", observed=True).apply(
    lambda g: pd.Series({
        "n_B": (g["cell_type"].isin(B_LINEAGE)).sum(),
        "n_T": (g["cell_type"].isin(T_LINEAGE)).sum(),
        "total": len(g),
    }),
    include_groups=False,
).join(sample_info)
bt_df["BT_ratio"] = bt_df["n_B"] / bt_df["n_T"].replace(0, np.nan)
bt_df["log2_BT"] = np.log2(bt_df["BT_ratio"].replace(0, np.nan))
bt_df["frac_B"] = bt_df["n_B"] / bt_df["total"]
bt_df["frac_T"] = bt_df["n_T"] / bt_df["total"]
bt_df["time_h"] = bt_df["time"].astype(str).str.replace("h", "").astype(int)

systems = sorted(bt_df["cell_system"].unique())
n_sys = len(systems)

# --- Row 1: log2(B/T) over time ---
fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, 5), sharey=True)
if n_sys == 1:
    axes = [axes]

for ax, sys in zip(axes, systems):
    sub = bt_df[bt_df["cell_system"] == sys].reset_index()
    for cond, color, marker in zip(["Mock", "Blinatumomab"],
                                    [COND_PALETTE.get("Mock", "#5975A4"), COND_PALETTE.get("Blinatumomab", "#E8655A")],
                                    ["o", "s"]):
        d = sub[sub["condition"] == cond]
        if d.empty:
            continue
        # Plot individual points
        ax.scatter(d["time_h"], d["log2_BT"], color=color, marker=marker,
                   s=60, zorder=3, edgecolors="white", linewidths=0.5)
        # Connect means
        means = d.groupby("time_h")["log2_BT"].mean()
        ax.plot(means.index, means.values, color=color, marker=marker,
                ls="-", lw=2, markersize=10, label=cond, zorder=2)

    ax.axhline(y=0, color="grey", ls="--", lw=1, alpha=0.5)
    ax.set_title(sys, fontweight="bold", fontsize=12)
    ax.set_xlabel("Time (h)")
    ax.set_xticks([6, 48])
    ax.set_ylabel("log\u2082(B / T)" if ax == axes[0] else "")
    if ax == axes[-1]:
        ax.legend(title="", frameon=True)
    sns.despine(ax=ax)

fig.suptitle("B/T ratio over time per cell system", fontweight="bold", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# --- Row 2: B and T fractions over time ---
fig, axes = plt.subplots(1, n_sys, figsize=(5 * n_sys, 5), sharey=True)
if n_sys == 1:
    axes = [axes]

for ax, sys in zip(axes, systems):
    sub = bt_df[bt_df["cell_system"] == sys].reset_index()
    for lineage, col, frac_col in [("B", "#4C72B0", "frac_B"), ("T", "#DD8452", "frac_T")]:
        for cond, ls in [("Mock", "-"), ("Blinatumomab", "--")]:
            d = sub[sub["condition"] == cond]
            if d.empty:
                continue
            means = d.groupby("time_h")[frac_col].mean()
            ax.plot(means.index, means.values, color=col, ls=ls, marker="o",
                    lw=2, markersize=8, label=f"{lineage} ({cond})")
            ax.scatter(d["time_h"], d[frac_col], color=col, alpha=0.3, s=25, zorder=1)

    ax.set_title(sys, fontweight="bold", fontsize=12)
    ax.set_xlabel("Time (h)")
    ax.set_xticks([6, 48])
    ax.set_ylabel("Fraction of cells" if ax == axes[0] else "")
    sns.despine(ax=ax)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, title="", frameon=True,
           bbox_to_anchor=(1.02, 0.5), loc="center left", fontsize=9)
fig.suptitle("B & T fractions over time — solid=Mock, dashed=Blinatumomab",
             fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Save annotated adata (end of section 7)
ANNOTATED_CACHE = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache") / "adata_annotated.h5ad"
ANNOTATED_CACHE.parent.mkdir(parents=True, exist_ok=True)
adata_filtered.write_h5ad(ANNOTATED_CACHE)
print(f"Saved annotated adata to {ANNOTATED_CACHE}")
print(adata_filtered)